In [ ]:
%%capture
!pip install huggingface_hub timm ipython -q

from kaggle_secrets import UserSecretsClient 
gh_token = UserSecretsClient().get_secret("GH_TOKEN")
!git clone https://{gh_token}@github.com/Hannysia/drone_detection.git
!pip install -e drone_detection/

%cd /kaggle/working/drone_detection

%load_ext autoreload
%autoreload 2

In [ ]:
import gc
from pathlib import Path
import torch
from huggingface_hub import HfApi

from src.config import REPO_ID, HF_TOKEN, EXTRACTED_DATA_DIR
from src.processing.embeddings import generate_and_save_embeddings

BATCH_SIZE = 64
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
if __name__ == "__main__":
    if not HF_TOKEN:
        raise ValueError("HF_TOKEN not found in Kaggle Secrets.")

    api = HfApi(token=HF_TOKEN)
    
    print("Loading DINOv2 ViT-B/14 model on GPU...")
    dinov2_model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14').to(DEVICE)
    dinov2_model.eval()
    
    TARGET_SUBDIR = "RAW_NEGATIVES"
    dataset_folder = EXTRACTED_DATA_DIR / "train"

    
    if not dataset_folder.exists():
        raise FileNotFoundError(f"Could not find dataset directory at {dataset_folder}.")

    emb_file_path = generate_and_save_embeddings(
        dataset_input_dir=dataset_folder, 
        model=dinov2_model,
        split_name="train",
        subdir_name=TARGET_SUBDIR,
        batch_size=BATCH_SIZE,
        device=DEVICE
    )
    
    print(f"\nUploading resulting embeddings file '{emb_file_path.name}' to Hugging Face...")
    api.upload_file(
        path_or_fileobj=str(emb_file_path),
        path_in_repo=f"embeddings/{emb_file_path.name}",
        repo_id=REPO_ID,
        repo_type="dataset",
        token=HF_TOKEN
    )
    
    print("\n" + "="*50)
    print("SUCCESS! EMBEDDINGS EXTRACTED AND UPLOADED TO HUGGING FACE!")
    print("="*50)

    del dinov2_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

Loading DINOv2 ViT-B/14 model on GPU...


Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main
Exception in thread QueueFeederThread:
Traceback (most recent call last):
  File "/usr/lib/python3.12/multiprocessing/queues.py", line 259, in _feed
Exception ignored in: <function _ConnectionBase.__del__ at 0x7f13f3e0f560>
Traceback (most recent call last):
  File "/usr/lib/python3.12/multiprocessing/connection.py", line 133, in __del__
    reader_close()
  File "/usr/lib/python3.12/multiprocessing/connection.py", line 178, in close
    self._close()
  File "/usr/lib/python3.12/multiprocessing/connection.py", line 377, in _close
    self._close()
  File "/usr/lib/python3.12/multiprocessing/connection.py", line 377, in _close
    _close(self._handle)
OSError: [Errno 9] Bad file descriptor

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    _close(self._handle)
OSError: [Errno 9] Bad fi


GENERATING EMBEDDINGS FOR: TRAIN (RAW_NEGATIVES)
Reading directly from: /kaggle/input/datasets/hannusiaknysh/train-drone-detection/extracted_dataset/train

Extracting embeddings via DINOv2 model...


DINOv2 Inference:   0%|          | 0/93 [00:00<?, ?it/s]


Saving embeddings dictionary to /kaggle/working/embeddings/train_negatives_embeddings.pt...

Uploading resulting embeddings file 'train_negatives_embeddings.pt' to Hugging Face...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            


SUCCESS! EMBEDDINGS EXTRACTED AND UPLOADED TO HUGGING FACE!
